# Integrate CITE and Xenium CD4 T cell data



- This notebook labels the activated CD4 T cell subset (from [02_label_refine.ipynb](02_label_refine.ipynb))in the Xenium dataset based on the CD4 subset labels in the CITE-seq dataset in this manuscript. 
- This analysis follows Seurat V5 Integration: https://satijalab.org/seurat/articles/integration_introduction
- First, we subset the CITE and Xenium datasets to only genes that are shared between modalitites and robustly expressed in the CITE dataset
- We then integrate the modalities according to the vignette above, using 'IntegrateLayers'
- We then cluster the integrated umap and label each cluster as the label of the most abundant CITE label in that cluster. 

**Pinned Environment:** [`conda_envs/r_seurat_20250604.yml`](../conda_envs/r_seurat_20250604.yml)

In [ ]:
library(Seurat)
library(harmony)
library(stringr)
library(patchwork)
library(dplyr)
library(tidyr)
library(cowplot)

In [ ]:
packageVersion("Seurat")

## Local file info

**Make sure to set** `DATA_DIR` in `config/paths.R` to point to the location of the downloaded CITEseq data.

In [ ]:
# source path config (edit config/paths.R to change locations)
source(file.path(dirname(getwd()), "config", "paths.R"))

# cite
cite_path <- file.path(DATA_DIR, "CITEseq", "D3_D30_20250519.Rds") # spatial_tls_manuscript_data/CITEseq_processed

# xenium 
adata_out_path <- file.path(BASE_OUTDIR, "cell_labeling/ouputs_mouselung_hurskainen_ref_consolidate_labels_xenseg_HDM/adata_outs/adata_cd4")


In [ ]:
# where to save outputs 
out_dir <- file.path(BASE_OUTDIR, "cell_labeling/ouputs_mouselung_hurskainen_ref_consolidate_labels_xenseg_HDM/cite_integration")
plot_out_dir = file.path(out_dir, 'plots')

if (!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)
if (!dir.exists(plot_out_dir)) dir.create(plot_out_dir, recursive = TRUE)

# Load seurat objects 

### CITE

In [ ]:
# main object
cite <- readRDS(cite_path)


In [ ]:
# Extract timepoint (D3, D5, D30)
cite$timepoint <- str_extract(cite$sample_name, "D30|D5|D3")
# Extract tissue (LN or Lung)
cite$tissue <- str_extract(cite$sample_name, "LN|Lung")

### Xenium 

In [ ]:
create_seurat_obj <- function(path){
    # Build a Seurat object from exported Xenium CSVs in `path`.
    # Reads counts.csv (transposed to genes x cells) and cell_metadata.csv and
    # constructs a Seurat object carrying that metadata.
    #
    # Args:
    #   path : directory containing "counts.csv" and "cell_metadata.csv".
    # Returns:
    #   A Seurat object with the loaded counts and cell metadata.

    # Load counts
    counts <- read.csv(file.path(path, "counts.csv"), row.names = 1)
    counts <- t(as.matrix(counts)) # transpose so that rows are genes and columns are cells
    
    # Load metadata
    meta <- read.csv(file.path(path,"cell_metadata.csv"), row.names = 1)
    
    # Create Seurat object
    so <- CreateSeuratObject(counts = counts, meta.data = meta)

    return(so)}

In [ ]:
xen <- create_seurat_obj(adata_out_path)


# Integrate

## subset to shared genes first

In [ ]:
shared_genes <- intersect(rownames(cite), rownames(xen))
cite <- subset(cite, features = shared_genes)
xen <- subset(xen, features = shared_genes)

print(dim(cite))
print(dim(xen))

# label modality
cite$modality <- 'cite'
xen$modality <- 'xen'

comb <- merge(cite, y=xen)

print(dim(comb))

## identify and subset to genes that are robustly expressed with the cite dataset

In [ ]:
# Get expression matrix (genes x cells)
expr_mat <- GetAssayData(cite, slot = "data")

# Extract cell annotations
annotations <- cite$cell_annotation
annotation_levels <- unique(annotations)

# Initialize a list to store expressed genes
expressed_genes_by_group <- list()

# Loop through each group and compute detection rate
for (group in annotation_levels) {
  group_cells <- which(annotations == group)
  group_expr <- expr_mat[, group_cells]
  detection_rate <- Matrix::rowSums(group_expr > 0) / length(group_cells)
  expressed_genes_by_group[[group]] <- names(detection_rate[detection_rate >= 0.05])
}

# Get the union of all expressed genes across any group
expressed_genes <- unique(unlist(expressed_genes_by_group))

# Optionally write to CSV
write.csv(expressed_genes, file = file.path(out_dir, "cite_expressed_genes_by_annotation.csv"), row.names = FALSE)


In [ ]:
comb <- subset(comb, features = expressed_genes)

## First run dim reduction without integration

In [ ]:
standard_dim_reduction <- function(obj, seed=1234){
    # Standard Seurat dimensionality-reduction pipeline (no integration).
    # Normalizes, finds variable features, scales, and runs PCA, then computes a
    # neighbor graph, Louvain clusters (stored as "unintegrated_clusters"), and a
    # UMAP ("umap.unintegrated") from PCA dims 1:10.
    #
    # Args:
    #   obj  : Seurat object (uses the RNA assay).
    #   seed : random seed for reproducibility (default 1234).
    # Returns:
    #   The Seurat object with PCA, unintegrated clusters, and the unintegrated UMAP added.

    DefaultAssay(obj) <- "RNA"

    set.seed(seed)
    
    obj <- NormalizeData(obj)
    obj <- FindVariableFeatures(obj)
    obj <- ScaleData(obj)
    obj <- RunPCA(obj)
    
    obj <- FindNeighbors(obj, dims = 1:10, reduction = "pca") #1:30
    obj <- FindClusters(obj, resolution = 1, cluster.name = "unintegrated_clusters")
    
    obj <- RunUMAP(obj, dims = 1:10, reduction = "pca", reduction.name = "umap.unintegrated") #1:30

    return(obj)}


In [ ]:
comb <- standard_dim_reduction(comb)

In [ ]:
DimPlot(comb, reduction = "umap.unintegrated", group.by = "modality")

## Run dim reduction with integration

In [ ]:
# now with integration
integrate_objects <- function(comb_obj, integration_method = CCAIntegration, reduction_name = 'integrated.cca', seed=1234){
    # Integrate the RNA layers of a merged Seurat object and recompute the embedding.
    # Runs Seurat's IntegrateLayers (default CCA) from the "pca" reduction into
    # `reduction_name`, rejoins the RNA layers, then computes a neighbor graph,
    # Louvain clusters (res 1), and a UMAP on the integrated reduction (dims 1:20).
    #
    # Args:
    #   comb_obj           : merged Seurat object with a "pca" reduction and split RNA layers.
    #   integration_method : Seurat integration method (default CCAIntegration).
    #   reduction_name     : name for the new integrated reduction (default 'integrated.cca').
    #   seed               : random seed for reproducibility (default 1234).
    # Returns:
    #   The integrated Seurat object with rejoined layers, clusters, and integrated UMAP.

    set.seed(seed)
    comb_obj <- IntegrateLayers(object = comb_obj, method = integration_method, orig.reduction = "pca", 
                                new.reduction = reduction_name,
                                verbose = FALSE)

    # print(ElbowPlot(comb_obj, reduction = "integrated.cca", ndims = 50))
    
    # re-join layers after integration
    comb_obj[["RNA"]] <- JoinLayers(comb_obj[["RNA"]])
    
    comb_obj <- FindNeighbors(comb_obj, reduction = reduction_name, dims = 1:20) # 1:30
    comb_obj <- FindClusters(comb_obj, resolution = 1)
    
    comb_obj <- RunUMAP(comb_obj, dims = 1:20, reduction = reduction_name, seed.use=seed) # 1:30
    return(comb_obj)}


In [ ]:
comb_cca <- integrate_objects(comb, integration_method = CCAIntegration, reduction_name = 'integrated.cca', seed=0001)

### Visualize integration results

In [ ]:
eval_plots <- function(obj) {

  print(DimPlot(obj, reduction = "umap", group.by = "modality"))
  print(DimPlot(obj, reduction = "umap", group.by = "cell_annotation", cols = "Set3"))
  print(DimPlot(obj, reduction = "umap", group.by = "cell_annotation", split.by = "modality", cols = "Set3"))

  # Genes to plot
  general <- c('Cd3e', 'Cd4', 'Cxcr6')
  Th0 <- c('Tcf7', 'Cxcr5')
  Th1 <- c('Tbx21', 'Ifng')
  Th2 <- c('Gata3', 'Il1rl1', 'Il5')
  Th17 <- c('Rorc', 'Il17a', 'Ccr6')
  Treg <- c('Foxp3')

  print(FeaturePlot(obj, features = general, reduction = "umap", split.by = 'modality'))
  print(FeaturePlot(obj, features = Th0, reduction = "umap", split.by = 'modality'))
  print(FeaturePlot(obj, features = Th1, reduction = "umap", split.by = 'modality'))
  print(FeaturePlot(obj, features = Th2, reduction = "umap", split.by = 'modality'))
  print(FeaturePlot(obj, features = Th17, reduction = "umap", split.by = 'modality'))
  print(FeaturePlot(obj, features = Treg, reduction = "umap", split.by = 'modality'))

}


In [ ]:
eval_plots(comb_cca)

## Label clusters

### Consolidate CITE labels

- Tfh and Th0 to Th0
- Th17 and Th17 Activated to Th17

In [ ]:
comb_cca$cell_annotation <- as.character(comb_cca$cell_annotation)
comb_cca$cell_annotation[comb_cca$cell_annotation == 'Tfh'] <- 'Th0'
comb_cca$cell_annotation <- factor(comb_cca$cell_annotation)

comb_cca$cell_annotation <- as.character(comb_cca$cell_annotation)
comb_cca$cell_annotation[comb_cca$cell_annotation == 'Th17 Activated'] <- 'Th17'
comb_cca$cell_annotation <- factor(comb_cca$cell_annotation)

In [ ]:
major_genes <- c('Tcf7','Tbx21', 'Il1rl1','Rorc','Foxp3')

In [ ]:
  # Genes to plot
  general <- c('Cd3e', 'Cd4', 'Cxcr6')
  Th0 <- c('Tcf7', 'Cxcr5')
  Th1 <- c('Tbx21', 'Ifng')
  Th2 <- c('Gata3', 'Il1rl1', 'Il5')
  Th17 <- c('Rorc', 'Il17a', 'Ccr6')
  Treg <- c('Foxp3', 'Il2ra')

In [ ]:
print(FeaturePlot(comb_cca, features = general, reduction = "umap"))
print(FeaturePlot(comb_cca, features = Th0, reduction = "umap"))
print(FeaturePlot(comb_cca, features = Th1, reduction = "umap"))
print(FeaturePlot(comb_cca, features = Th2, reduction = "umap"))
print(FeaturePlot(comb_cca, features = Th17, reduction = "umap"))
print(FeaturePlot(comb_cca, features = Treg, reduction = "umap"))

### Update clustering resolution

In [ ]:
comb_cca <- FindClusters(comb_cca, resolution = 1.2, cluster.name = "seurat_clusters")

In [ ]:
# palette
library(RColorBrewer)
cluster_ids <- sort(unique(comb_cca$seurat_clusters))
# palette_colors <- brewer.pal(n = length(cluster_ids), name = "Paired")
palette_colors <- pal_d3("category20")(length(cluster_ids))

# Make a named vector
cluster_colors <- setNames(palette_colors, cluster_ids)

In [ ]:
DimPlot(comb_cca,
               reduction = "umap", group.by = "seurat_clusters", split.by = "modality", cols=cluster_colors)
DimPlot(comb_cca,
               reduction = "umap", group.by = "cell_annotation", split.by = "modality")
FeaturePlot(comb_cca,
               reduction = "umap", feature = "Tcf7", split.by = "modality")

### Find cluster markers

In [ ]:
Idents(comb_cca) <- "seurat_clusters"
cluster.markers <- FindAllMarkers(comb_cca, only.pos = TRUE, min.pct = 0.25, logfc.threshold = 0.25)
head(cluster.markers)


In [ ]:
write.csv(cluster.markers, file = file.path(out_dir, "integrated_cluster_markers.csv"), row.names = FALSE)

In [ ]:
top_markers <- cluster.markers %>%
  group_by(cluster) %>%
  top_n(n = 15, wt = avg_log2FC)

top_genes <- unique(top_markers$gene)

p<- DotPlot(comb_cca, features = top_genes, group.by = "seurat_clusters") +
  RotatedAxis()
# p

### Assign each cluster name based on highest fraction cite label

In [ ]:
# Subset to CITE cells only
cite_only <- comb_cca@meta.data %>%
  filter(modality == "cite", !is.na(cell_annotation))


# Create wide table
cluster_label_table <- cite_only %>%
  group_by(seurat_clusters, cell_annotation) %>%
  summarise(count = n(), .groups = "drop") %>%
  pivot_wider(
    names_from = cell_annotation,
    values_from = count,
    values_fill = 0
  )

# Get label columns only (exclude seurat_clusters)
label_cols <- setdiff(names(cluster_label_table), "seurat_clusters")

# Compute max label and fraction
cluster_label_table <- cluster_label_table %>%
  rowwise() %>%
  mutate(
    total_cells = sum(c_across(all_of(label_cols))),
    max_label = label_cols[which.max(c_across(all_of(label_cols)))],
    max_fraction = max(c_across(all_of(label_cols))) / total_cells
  ) %>%
  ungroup()

# Create named vectors
cluster_label_map <- setNames(cluster_label_table$max_label, cluster_label_table$seurat_clusters)
cluster_fraction_map <- setNames(cluster_label_table$max_fraction, cluster_label_table$seurat_clusters)

# Add to Seurat metadata

comb_cca$integrated_cluster <- NA
for (cluster in names(cluster_label_map)) {
  comb_cca$integrated_cluster[WhichCells(comb_cca, idents = cluster)] <- cluster_label_map[[cluster]]
}

In [ ]:
cluster_label_table

### For cluster with too few matching cells in CITE, manually label based on markers
- Label this cluster as 'CD4 Trans' for transitioning CD4; it has markers of cell cycle and adhesion and does not clearly fit into one of the other labels

In [ ]:
top_markers %>% filter(cluster == '1')

In [ ]:
comb_cca$integrated_cluster <- as.character(comb_cca$integrated_cluster)
comb_cca$integrated_cluster[comb_cca$seurat_clusters == '1'] <- 'CD4 trans'
comb_cca$integrated_cluster <- factor(comb_cca$integrated_cluster)

### Plot final labels

In [ ]:
# palette
library(RColorBrewer)
ids <- sort(unique(c(as.character(comb_cca$cell_annotation), as.character(comb_cca$integrated_cluster))))
palette_colors <- brewer.pal(n = length(cluster_ids), name = "Paired")

# Make a named vector
colors <- setNames(palette_colors, ids)

In [ ]:
DimPlot(comb_cca, reduction = "umap", group.by = "integrated_cluster", split.by = "modality", cols=colors)
DimPlot(comb_cca, reduction = "umap", group.by = "cell_annotation", split.by = "modality", cols=colors)

In [ ]:
DimPlot(comb_cca, reduction = "umap", group.by = "integrated_cluster", split.by = "modality", cols=colors)
DimPlot(comb_cca, reduction = "umap", group.by = "cell_annotation", split.by = "modality", cols=colors)

In [ ]:
FeaturePlot(comb_cca, features = 'Tcf7', reduction = "umap", split.by = "modality")
FeaturePlot(comb_cca, features = 'Slamf6', reduction = "umap", split.by = "modality")
FeaturePlot(comb_cca, features = 'Cxcr5', reduction = "umap", split.by = "modality")
FeaturePlot(comb_cca, features = 'Rorc', reduction = "umap", split.by = "modality")
FeaturePlot(comb_cca, features = 'Tbx21', reduction = "umap", split.by = "modality")

# Export object data for loading into python

In [ ]:
export_seurat_for_adata <- function(seurat_obj, output_path, umap_name = "umap") {
  # Export a Seurat object to CSVs for loading into Python/AnnData.
  # Writes the RNA "data" matrix (counts.csv), cell metadata (cell_metadata.csv),
  # and, if present, the named UMAP embedding (<umap_name>.csv) into `output_path`
  # (created if it does not exist).
  #
  # Args:
  #   seurat_obj  : Seurat object to export.
  #   output_path : directory to write the CSVs into (created if missing).
  #   umap_name   : reduction name to export coordinates for (default "umap");
  #                 skipped with a warning if not present in the object.
  # Returns:
  #   Nothing; writes files as a side effect.

  # Create output directory if it doesn't exist
  if (!dir.exists(output_path)) {
    dir.create(output_path, recursive = TRUE)
  }

  # Export counts matrix
  counts_mat <- GetAssayData(seurat_obj, assay = "RNA", slot = "data")
  counts_df <- as.data.frame(as.matrix(counts_mat))
  write.csv(counts_df, file = file.path(output_path, "counts.csv"))

  # Export cell metadata
  meta_df <- seurat_obj@meta.data
  write.csv(meta_df, file = file.path(output_path, "cell_metadata.csv"))

  # Export UMAP coordinates
  if (umap_name %in% names(seurat_obj@reductions)) {
    umap_df <- as.data.frame(Embeddings(seurat_obj, reduction = umap_name))
    write.csv(umap_df, file = file.path(output_path, paste0(umap_name, ".csv")))
  } else {
    warning(paste("UMAP reduction", umap_name, "not found in Seurat object. Skipping UMAP export."))
  }
}

In [ ]:
output_path = file.path(out_dir, 'seurat_obj_outs', 'cite_xen_integrated')

In [ ]:
export_seurat_for_adata(comb_cca, output_path)

In [ ]:
env_path <- normalizePath(R.home())
env_name <- basename(dirname(dirname(env_path)))
cat("Active conda environment:", env_name, "\n")

In [ ]:
sessionInfo()